In [0]:
try:

    # ----------------------------------------------------
    # 1. Find newly ingested records
    # ----------------------------------------------------

    new_arrivals = (
        spark.table(SOURCE_TABLE)
        .filter(
            (F.col("ingestion_ts") > F.lit(last_watermark))
            &
            (F.col("ingestion_ts") <= F.lit(run_upper_bound))
        )
    )

    affected_input_dates = (
        new_arrivals
        .select("event_date")
        .distinct()
    )


    # ----------------------------------------------------
    # 2. Expand dirty output scope
    # ----------------------------------------------------

    dirty_output_dates = (
        affected_input_dates
        .select(
            F.explode(
                F.sequence(
                    F.col("event_date"),
                    F.least(
                        F.date_add(
                            F.col("event_date"),
                            ROLLING_DAYS - 1
                        ),
                        F.lit(run_date)
                    )
                )
            ).alias("event_date")
        )
        .distinct()
    )


    # ----------------------------------------------------
    # 3. Determine required source history
    # ----------------------------------------------------

    required_source_dates = (
        dirty_output_dates
        .select(
            F.explode(
                F.sequence(
                    F.date_sub(
                        F.col("event_date"),
                        ROLLING_DAYS - 1
                    ),
                    F.col("event_date")
                )
            ).alias("event_date")
        )
        .distinct()
    )


    # ----------------------------------------------------
    # 4. Read complete trusted source
    # ----------------------------------------------------

    source_events = (
        spark.table(SOURCE_TABLE)
        .join(
            F.broadcast(required_source_dates),
            "event_date",
            "left_semi"
        )
        .select(
            "event_id",
            "event_date",
            "product_id",
            "market",
            "event_type",
            "amount"
        )
    )


    # ----------------------------------------------------
    # 5. Deterministic business calculation
    # ----------------------------------------------------

    daily_metrics = (
        source_events
        .groupBy(
            "event_date",
            "product_id",
            "market"
        )
        .agg(
            F.sum(
                F.when(
                    F.col("event_type") == "view",
                    1
                ).otherwise(0)
            ).alias("daily_views"),

            F.sum(
                F.when(
                    F.col("event_type") == "purchase",
                    1
                ).otherwise(0)
            ).alias("daily_purchases"),

            F.sum(
                F.when(
                    F.col("event_type") == "purchase",
                    F.coalesce("amount", F.lit(0.0))
                ).otherwise(0.0)
            ).alias("daily_revenue")
        )
        .withColumn(
            "day_number",
            F.datediff(
                "event_date",
                F.lit("1970-01-01")
            )
        )
    )


    rolling_window = (
        Window
        .partitionBy(
            "product_id",
            "market"
        )
        .orderBy("day_number")
        .rangeBetween(
            -(ROLLING_DAYS - 1),
            0
        )
    )


    recalculated = (
        daily_metrics

        .withColumn(
            "views_7d",
            F.sum("daily_views").over(rolling_window)
        )

        .withColumn(
            "purchases_7d",
            F.sum("daily_purchases").over(rolling_window)
        )

        .withColumn(
            "revenue_7d",
            F.sum("daily_revenue").over(rolling_window)
        )

        .withColumn(
            "conversion_rate_7d",
            F.when(
                F.col("views_7d") > 0,
                F.col("purchases_7d")
                / F.col("views_7d")
            )
        )
    )


    # ----------------------------------------------------
    # 6. Keep only dirty output slices
    # ----------------------------------------------------

    replacement_df = (
        recalculated
        .join(
            F.broadcast(dirty_output_dates),
            "event_date",
            "left_semi"
        )
        .select(
            "event_date",
            "product_id",
            "market",
            "daily_views",
            "daily_purchases",
            "daily_revenue",
            "views_7d",
            "purchases_7d",
            "revenue_7d",
            "conversion_rate_7d"
        )
    )


    # ----------------------------------------------------
    # 7. Validate
    # ----------------------------------------------------

    duplicates = (
        replacement_df
        .groupBy(
            "event_date",
            "product_id",
            "market"
        )
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicates.limit(1).count() > 0:
        raise ValueError(
            "Duplicate target grain detected"
        )


    # ----------------------------------------------------
    # 8. Construct selective overwrite predicate
    # ----------------------------------------------------

    dirty_dates = [
        x["event_date"]
        for x in dirty_output_dates.collect()
    ]

    date_literals = ", ".join(
        f"DATE '{d.isoformat()}'"
        for d in dirty_dates
    )

    replace_predicate = (
        f"event_date IN ({date_literals})"
    )


    # ----------------------------------------------------
    # 9. Atomic selective overwrite
    # ----------------------------------------------------

    (
        replacement_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            replace_predicate
        )
        .saveAsTable(TARGET_TABLE)
    )


    # ----------------------------------------------------
    # 10. Advance watermark only after successful write
    # ----------------------------------------------------

    (
        spark.createDataFrame(
            [(
                run_id,
                last_watermark,
                run_upper_bound,
                len(dirty_dates),
                "SUCCESS",
                None
            )],
            """
            run_id STRING,
            watermark_from TIMESTAMP,
            watermark_to TIMESTAMP,
            affected_output_dates LONG,
            status STRING,
            error_message STRING
            """
        )
        .withColumn(
            "affected_input_dates",
            F.lit(None).cast("long")
        )
        .withColumn(
            "completed_at",
            F.current_timestamp()
        )
        .select(
            "run_id",
            "watermark_from",
            "watermark_to",
            "affected_input_dates",
            "affected_output_dates",
            "status",
            "error_message",
            "completed_at"
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(CONTROL_TABLE)
    )

except Exception as e:

    (
        spark.createDataFrame(
            [(
                run_id,
                last_watermark,
                run_upper_bound,
                0,
                0,
                "FAILED",
                str(e)[:4000]
            )],
            """
            run_id STRING,
            watermark_from TIMESTAMP,
            watermark_to TIMESTAMP,
            affected_input_dates LONG,
            affected_output_dates LONG,
            status STRING,
            error_message STRING
            """
        )
        .withColumn(
            "completed_at",
            F.current_timestamp()
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(CONTROL_TABLE)
    )

    raise